In [1]:
%run ../setup_env.py

Checking environment...
  ✓ faiss
  ✓ datasets


2026-05-25 14:34:27.174615: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.

  ✓ evaluate
  ✓ accelerate
  ✓ sentencepiece

All packages present — ready to go
FAISS patch applied — version 1.14.1


In [2]:
import os
import sys
import json
import faiss
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import transformers.utils.import_utils as import_utils
import transformers.utils as tu

# faiss patch
if hasattr(import_utils.is_faiss_available, "cache_clear"):
    import_utils.is_faiss_available.cache_clear()
import_utils._faiss_available = True
import_utils.is_faiss_available = lambda: True
tu.is_faiss_available = lambda: True

# paths
REPO_ROOT    = os.path.abspath(os.path.join(os.getcwd(), "../.."))
FEVER_DIR    = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR     = os.path.join(FEVER_DIR, "data")
RESULTS_DIR  = os.path.join(FEVER_DIR, "results")
CONFIG_DIR  = os.path.join(FEVER_DIR, "configs")

os.makedirs(RESULTS_DIR, exist_ok=True)

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Fever dir:   {FEVER_DIR}")
print(f"Data dir:    {DATA_DIR}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Config dir: {CONFIG_DIR}")
print(f"FAISS patch: {import_utils.is_faiss_available()}")

Fever dir:   /home/jovyan/lectures/raq-reproducibility-challenge/fever
Data dir:    /home/jovyan/lectures/raq-reproducibility-challenge/fever/data
Results dir: /home/jovyan/lectures/raq-reproducibility-challenge/fever/results
Config dir: /home/jovyan/lectures/raq-reproducibility-challenge/fever/configs
FAISS patch: True


In [3]:
from datasets import load_dataset
import yaml

# load passages
print("Loading passages...")
passages = []
with open(os.path.join(DATA_DIR, "fever_passages.jsonl")) as f:
    for line in f:
        passages.append(json.loads(line))
print(f"Passages loaded: {len(passages):,}")

# load faiss index
print("Loading FAISS index...")
index = faiss.read_index(
    os.path.join(DATA_DIR, "fever_faiss.index")
)
print(f"Index loaded:    {index.ntotal:,} vectors")

# load fever dataset
print("Loading FEVER dataset...")
dataset = load_dataset("copenlu/fever_gold_evidence")
print(f"Dataset loaded:  {dataset}")

# label mapping
LABEL2ID = {
    "SUPPORTS":        "0",
    "REFUTES":         "1",
    "NOT ENOUGH INFO": "2"
}

# Training subset size
with open(CONFIG_DIR + "/fever_config.yaml","r") as file:
    config = yaml.safe_load(file)
    
TRAINING_SIZE = config["data"]["train_size"]
print(f"Training size: {TRAINING_SIZE}") 

def clean_fever_title(title):
    title = title.replace("-LRB-", "(")
    title = title.replace("-RRB-", ")")
    title = title.replace("-LSB-", "[")
    title = title.replace("-RSB-", "]")
    title = title.replace("-LCB-", "{")
    title = title.replace("-RCB-", "}")
    title = title.replace("_", " ")
    return title.strip()

Loading passages...
Passages loaded: 574,197
Loading FAISS index...
Index loaded:    574,197 vectors
Loading FEVER dataset...
Dataset loaded:  DatasetDict({
    train: Dataset({
        features: ['claim', 'label', 'evidence', 'id', 'verifiable', 'original_id'],
        num_rows: 228277
    })
    validation: Dataset({
        features: ['claim', 'label', 'evidence', 'id', 'verifiable', 'original_id'],
        num_rows: 15935
    })
    test: Dataset({
        features: ['claim', 'label', 'evidence', 'id', 'verifiable', 'original_id'],
        num_rows: 16039
    })
})
Training size: 500


In [4]:
from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizerFast
)

print("Loading DPR question encoder...")
q_tokenizer = DPRQuestionEncoderTokenizerFast.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
)
q_encoder = DPRQuestionEncoder.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
).to("cuda")
q_encoder.train()
print(f"Question encoder loaded on: "
      f"{next(q_encoder.parameters()).device}")

Loading DPR question encoder...


Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Question encoder loaded on: cuda:0


In [5]:
from transformers import(
    BartForConditionalGeneration,
    BartTokenizer)

print("Loading BART model and tokenizer...")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large").to("cuda")
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")

model.train()

print(f"BART model loaded on: "
      f"{next(model.parameters()).device}")


Loading BART model and tokenizer...
BART model loaded on: cuda:0


In [6]:
allocated = torch.cuda.memory_allocated() / 1024**3
reserved  = torch.cuda.memory_reserved() / 1024**3
total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\nVRAM usage after model loading:")
print(f"  Allocated: {allocated:.1f} GB")
print(f"  Reserved:  {reserved:.1f} GB")
print(f"  Total:     {total:.1f} GB")
print(f"  Free:      {total - reserved:.1f} GB")


VRAM usage after model loading:
  Allocated: 1.9 GB
  Reserved:  2.0 GB
  Total:     44.4 GB
  Free:      42.5 GB


In [7]:
LABEL_TOKEN_IDS = [288, 134, 176]  # token ids for "0", "1", "2"
N_DOCS = config["training"]["n_docs"]
MAX_LENGTH = config["training"]["max_source_length"]

def search_index(claim, index, passages, q_encoder,
                 q_tokenizer, n_docs=N_DOCS, training=False):
    """
    Encode a claim with DPR and retrieve top-n passages
    from the FAISS index.
    """
    encoded = q_tokenizer(
        claim,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )
    if training:
        query_vec = q_encoder(
            input_ids=encoded["input_ids"].to("cuda"),
            attention_mask=encoded["attention_mask"].to("cuda")
        ).pooler_output
        query_vec_np = query_vec.detach().cpu().numpy()
    else:
        with torch.no_grad():
            query_vec = q_encoder(
                input_ids=encoded["input_ids"].to("cuda"),
                attention_mask=encoded["attention_mask"].to("cuda")
            ).pooler_output
        query_vec_np = query_vec.cpu().numpy()

    query_vec_np_norm = query_vec_np.copy()
    faiss.normalize_L2(query_vec_np_norm)
    scores, indices = index.search(
        query_vec_np_norm.astype("float32"), n_docs
    )

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "rank":  len(results) + 1,
            "score": float(score),
            "title": passages[idx]["title"],
            "text":  passages[idx]["text"],
            "idx":   int(idx)
        })
    return results, query_vec if training else None

print("Search function defined")

def prepare_bart_inputs(claim, retrieved_passages, bart_tokenizer, max_lengh):
    texts = []
    for passage in retrieved_passages:
        text = f"question: {claim} title: {passage['title']} context: {passage['text']}"
        texts.append(text)

    encoded = bart_tokenizer(
        texts,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    return encoded

def forward_pass(claim, retrieved_results, gold_label_str,
                 model, bart_tokenizer, label2id):

    # 1. prepare BART inputs
    encoded = prepare_bart_inputs(
        claim,
        retrieved_results,
        bart_tokenizer,
        MAX_LENGTH
    )

    # 2. retrieval scores
    retrieval_scores = [r["score"] for r in retrieved_results]

    # 3. prepare labels — gold token repeated K times
    K             = len(retrieved_results)
    gold_token_id = LABEL_TOKEN_IDS[int(label2id[gold_label_str])]
    labels        = torch.full((K, 1), gold_token_id)

    # 4. BART forward pass
    output = model(
        input_ids      = encoded["input_ids"].to("cuda"),
        attention_mask = encoded["attention_mask"].to("cuda"),
        labels         = labels.to("cuda")
    )

    # 5. extract label logits at first token position
    # output.logits shape: [K, seq_len, vocab_size]
    logits       = output.logits[:, 0, :]        # [K, vocab_size]
    label_logits = logits[:, LABEL_TOKEN_IDS]    # [K, 3]

    # 6. retrieval probs
    retrieval_probs = torch.softmax(
        torch.tensor(retrieval_scores).to("cuda"),
        dim=0
    )                                             # [K]

    # 7. marginalize
    marginalized = (
        retrieval_probs.unsqueeze(1) * label_logits
    ).sum(dim=0)                                  # [3]

    return marginalized, output.loss

Search function defined


In [25]:
def evaluate(data, model, q_encoder, bart_tokenizer,
             q_tokenizer, index, passages, label2id):
    """
    Run forward pass on data without gradients.
    Returns 3-way accuracy and 2-way accuracy.
    """
    model.eval()
    q_encoder.eval()

    correct_3way = 0
    correct_2way = 0
    total_3way = 0
    total_2way = 0

    id2label = {v: k for k, v in label2id.items()}

    with torch.no_grad():
        for example in data:

            # retrieve
            results, _ = search_index(
                example["claim"],
                index,
                passages,
                q_encoder,
                q_tokenizer,
                n_docs=config["model"]["n_docs"],
                training=False
            )

            # forward pass
            marginalized, _ = forward_pass(
                example["claim"],
                results,
                example["label"],
                model,
                bart_tokenizer,
                label2id
            )

            # predict
            pred_idx   = torch.argmax(marginalized).item()
            pred_label = id2label[str(pred_idx)]
            gold_label = example["label"]

            # 3-way accuracy
            total_3way   += 1
            correct_3way += (pred_label == gold_label)

            # 2-way accuracy — skip NEI examples
            if gold_label != "NOT ENOUGH INFO":
                total_2way   += 1
                correct_2way += (pred_label == gold_label)

    acc_3way = correct_3way / total_3way
    acc_2way = correct_2way / max(total_2way, 1)

    model.train()
    q_encoder.train()

    return acc_3way, acc_2way

In [ ]:
from torch.amp import GradScaler
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

def train(model, q_encoder, train_data, val_data):

    # --- setup inside the function ---
    GRAD_ACCUM = config["training"]["gradient_accumulation_steps"]
    EPOCHS     = config["training"]["epochs"]

    total_steps = (
        len(train_data) // config["training"]["batch_size"]
    ) * EPOCHS

    optimizer = AdamW([
        {"params": q_encoder.parameters(),
         "lr": config["training"]["learning_rate"]},
        {"params": model.parameters(),
         "lr": config["training"]["learning_rate"]}
    ], weight_decay=config["training"]["weight_decay"])

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=config["training"]["warmup_steps"],
        num_training_steps=total_steps
    )

    scaler = GradScaler()

    loss_fn = torch.nn.CrossEntropyLoss(
        label_smoothing=config["training"]["label_smoothing"]
    )

    CHECKPOINT_DIR = os.path.join(FEVER_DIR, "results", "checkpoints")
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    best_val_acc = 0.0

    # --- epoch loop ---
    for epoch in range(EPOCHS):

        model.train()
        q_encoder.train()

        total_loss = 0
        correct    = 0
        total      = 0

        optimizer.zero_grad()

        for i, example in enumerate(train_data):

            # retrieve
            retrieved_results, _ = search_index(
                example["claim"], index, passages,
                q_encoder, q_tokenizer,
                n_docs=config["model"]["n_docs"],
                training=True
            )

            # forward pass
            marginalized, _ = forward_pass(
                example["claim"],
                retrieved_results,
                example["label"],
                model,
                bart_tokenizer,
                LABEL2ID
            )

            # loss
            gold_idx = torch.tensor(
                [int(LABEL2ID[example["label"]])]
            ).to("cuda")
            loss = loss_fn(marginalized.unsqueeze(0), gold_idx)
            loss = loss / GRAD_ACCUM

            # backward
            scaler.scale(loss).backward()

            # optimizer step every GRAD_ACCUM iterations
            if (i + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    list(model.parameters()) +
                    list(q_encoder.parameters()),
                    config["training"]["max_grad_norm"]
                )
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            # metrics
            total_loss += loss.item() * GRAD_ACCUM
            pred        = torch.argmax(marginalized).item()
            correct    += int(
                pred == int(LABEL2ID[example["label"]])
            )
            total += 1

            # log
            if i % 50 == 0:
                print(f"Epoch {epoch+1} | "
                      f"Step {i}/{len(train_data)} | "
                      f"Loss: {total_loss/(i+1):.4f} | "
                      f"Acc: {correct/max(total,1):.1%}")

        # validation
        val_acc_3way, val_acc_2way = evaluate(
            val_data, model, q_encoder,
            bart_tokenizer, q_tokenizer,
            index, passages, LABEL2ID
        )

        print(f"\nEpoch {epoch+1} complete")
        print(f"  Train acc:     {correct/total:.1%}")
        print(f"  Val 3-way acc: {val_acc_3way:.1%}  (target 72.5%)")
        print(f"  Val 2-way acc: {val_acc_2way:.1%}  (target 89.5%)")

        torch.save(
            q_encoder.state_dict(),
            f"{CHECKPOINT_DIR}/q_encoder_epoch{epoch+1}.pt"
        )
        torch.save(
            model.state_dict(),
            f"{CHECKPOINT_DIR}/bart_epoch{epoch+1}.pt"
        )
        
        if val_acc_3way > best_val_acc:
            best_val_acc = val_acc_3way
            torch.save(
                q_encoder.state_dict(),
                f"{CHECKPOINT_DIR}/q_encoder_best.pt"
            )
            torch.save(
                model.state_dict(),
                f"{CHECKPOINT_DIR}/bart_best.pt"
            )
        print(f"  New best saved — acc: {best_val_acc:.1%}")

In [ ]:
random.seed(config["data"]["seed"])
random.shuffle(full_train)

# full dataset
train_data = full_train          # all 228K examples
val_data_small = random.sample(
    list(dataset["validation"]), 500
)

print(f"Train: {len(train_data):,} | Val: {len(val_data_small):,}")
print("Starting full training — this will run overnight...")
print("Expected: ~2-4 hours per epoch on A40\n")
best = train(model, q_encoder, train_data, val_data_small)

Train: 228,277 | Val: 500
Starting full training — this will run overnight...
Expected: ~2-4 hours per epoch on A40

Epoch 1 | Step 0/228277 | Loss: 1.2022 | Acc: 0.0%
Epoch 1 | Step 50/228277 | Loss: 1.0380 | Acc: 45.1%
Epoch 1 | Step 100/228277 | Loss: 1.0437 | Acc: 42.6%
Epoch 1 | Step 150/228277 | Loss: 0.9942 | Acc: 46.4%
Epoch 1 | Step 200/228277 | Loss: 0.9817 | Acc: 48.3%
Epoch 1 | Step 250/228277 | Loss: 0.9673 | Acc: 49.4%
Epoch 1 | Step 300/228277 | Loss: 0.9645 | Acc: 50.5%
Epoch 1 | Step 350/228277 | Loss: 0.9500 | Acc: 53.6%
Epoch 1 | Step 400/228277 | Loss: 0.9612 | Acc: 53.4%
Epoch 1 | Step 450/228277 | Loss: 0.9400 | Acc: 55.0%
Epoch 1 | Step 500/228277 | Loss: 0.9231 | Acc: 56.3%
Epoch 1 | Step 550/228277 | Loss: 0.9221 | Acc: 56.8%
Epoch 1 | Step 600/228277 | Loss: 0.9199 | Acc: 57.4%
Epoch 1 | Step 650/228277 | Loss: 0.9163 | Acc: 57.9%
Epoch 1 | Step 700/228277 | Loss: 0.9231 | Acc: 57.8%
Epoch 1 | Step 750/228277 | Loss: 0.9216 | Acc: 58.1%
Epoch 1 | Step 800/2282

In [ ]:
import time
import torch

example = list(dataset["train"])[0]
print("yes")

# time each component separately
t0 = time.time()
results, _ = search_index(
    example["claim"], gpu_index, passages,
    q_encoder, q_tokenizer, training=True
)
t1 = time.time()
print(f"Retrieval (GPU):    {(t1-t0)*1000:.1f} ms")

from data import prepare_bart_inputs
encoded = prepare_bart_inputs(
    example["claim"], results,
    bart_tokenizer, 300
)
t2 = time.time()
print(f"Tokenization:       {(t2-t1)*1000:.1f} ms")

gold_token_id = LABEL_TOKEN_IDS[0]
K      = len(results)
labels = torch.full((K, 1), gold_token_id)
output = model(
    input_ids      = encoded["input_ids"].to("cuda"),
    attention_mask = encoded["attention_mask"].to("cuda"),
    labels         = labels.to("cuda")
)
t3 = time.time()
print(f"BART forward pass:  {(t3-t2)*1000:.1f} ms")

print(f"\nTotal per step:     {(t3-t0)*1000:.1f} ms")
print(f"Projected epoch:    {(t3-t0) * 228277 / 3600:.1f} hours")